# E04

In [ ]:

from __future__ import annotations

import csv
import json
import math
import sys
from copy import deepcopy
from pathlib import Path


import shutil
import tempfile
import zipfile

SOURCE_ZIP_NAME = 'syniscopy_source.zip'
IN_COLAB = Path('/content').exists()
DRIVE_MYDRIVE = Path('/content/drive/MyDrive')
if IN_COLAB and not DRIVE_MYDRIVE.exists():
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as exc:
        print('Drive was not mounted automatically:', repr(exc))


def find_supplemental_root() -> Path:
    explicit = globals().get('SYNISCOPY_SUPPLEMENTAL_ROOT', None)
    candidates = []
    if explicit:
        candidates.append(Path(explicit).expanduser())
    here = Path.cwd().resolve()
    candidates.extend([here, *here.parents])
    if DRIVE_MYDRIVE.exists():
        candidates.extend([
            DRIVE_MYDRIVE / 'supplemental',
            DRIVE_MYDRIVE / 'SyniscopySupplemental',
        ])
    candidates.extend([
        Path('/content/supplemental'),
        Path('/content/SyniscopySupplemental'),
    ])
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            resolved = candidate
        if (resolved / 'supplemental' / 'E01.ipynb').exists():
            resolved = resolved / 'supplemental'
        if (resolved / 'E01.ipynb').exists() and (
            (resolved / SOURCE_ZIP_NAME).exists() or (resolved.parent / 'codebase').is_dir()
        ):
            return resolved
    raise RuntimeError(
        'Syniscopy supplemental folder not found. Upload the supplemental folder to Drive as MyDrive/supplemental, '
        'including syniscopy_source.zip.'
    )


def prepare_syniscopy_source(supplemental_root: Path) -> Path:
    source_root = supplemental_root.parent
    if (source_root / 'codebase').is_dir():
        return source_root
    zip_path = supplemental_root / SOURCE_ZIP_NAME
    if not zip_path.exists():
        raise FileNotFoundError(
            f'Missing {zip_path}. Run python supplemental/package_experiments_for_colab.py before uploading supplemental/.'
        )
    extract_dir = Path('/content/syniscopy_source') if IN_COLAB else Path(tempfile.gettempdir()) / 'syniscopy_source'
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    if (extract_dir / 'codebase').is_dir():
        return extract_dir
    candidates = [p for p in extract_dir.iterdir() if p.is_dir() and (p / 'codebase').is_dir()]
    if len(candidates) != 1:
        raise RuntimeError(f'Could not find a single codebase/ root after extracting {zip_path}.')
    return candidates[0]


SUPPLEMENTAL_ROOT = find_supplemental_root()
ROOT = prepare_syniscopy_source(SUPPLEMENTAL_ROOT)
CODEBASE = ROOT / 'codebase'
OUTPUT_DIR = SUPPLEMENTAL_ROOT / 'outputs' / 'E04'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if str(CODEBASE) not in sys.path:
    sys.path.insert(0, str(CODEBASE))

import numpy as np

from config import PARAMS
from fisher_diagnostic import (
    compare_modality_axial_information_content,
    compare_modality_information_content,
    compare_modality_information_content_detected_quanta_normalized,
    compute_fisher_information,
    compute_localization_crlb,
    compute_localization_crlb_3d,
    compute_modality_fusion_crlb,
)
from calibration_profiles import write_calibration_outputs
from camera_noise import analysis_contrast_noise_variance
from imaging_model import modality_display_name, modality_uses_relative_reference_contrast
from main import generate_single_frame_views

HEADLINE_MODALITIES = [
    'bright_field',
    'fluorescence_widefield',
    'tirf_fluorescence',
    'dark_field',
    'zernike_phase_contrast',
    'differential_phase_contrast',
    'quantitative_phase',
    'off_axis_holography',
    'ricm',
    'interferometric',
    'tem_phase_contrast',
    'sem_secondary_electron',
]
PANEL_MODALITIES = [
    'bright_field',
    'fluorescence_widefield',
    'tirf_fluorescence',
    'dark_field',
    'zernike_phase_contrast',
    'differential_phase_contrast',
    'quantitative_phase',
    'off_axis_holography',
    'ricm',
    'interferometric',
    'tem_phase_contrast',
    'sem_secondary_electron',
]

IMAGE_SIZE = 192
PUPIL_SAMPLES = 192
PIXEL_SIZE_NM = 65.0
WAVELENGTH_NM = 532.0
DIAMETER_NM = 100.0
Z_NM = 0.0
Z_STEP_NM = 200.0
NOISE_VARIANCE = 1.0
QUANTA_BUDGET = 5e4
BUDGET_READOUT_VARIANCE = 1.0




def finite_or_inf(value: float, digits: int = 3) -> str:
    value = float(value)
    if not np.isfinite(value):
        return '$\\infty$'
    return f'{value:.{digits}g}'


def stable_seed(label: str) -> int:
    seed = 0
    for index, char in enumerate(str(label)):
        seed = (seed + (index + 1) * ord(char)) % (2**32)
    return seed


def display_from_contrast(contrast: np.ndarray, label: str, display_noise_std: float = 0.01) -> np.ndarray:
    image = np.asarray(contrast, dtype=float)
    finite = image[np.isfinite(image)]
    if finite.size == 0:
        return np.zeros(image.shape, dtype=np.uint8)
    center = float(np.median(finite))
    deviation = np.where(np.isfinite(image), image - center, 0.0)
    finite_deviation = deviation[np.isfinite(deviation)]
    scale = float(np.percentile(np.abs(finite_deviation), 99.5)) if finite_deviation.size else 0.0
    if not np.isfinite(scale) or scale <= 0.0:
        scale = float(np.max(np.abs(finite_deviation))) if finite_deviation.size else 0.0
    if not np.isfinite(scale) or scale <= 0.0:
        return np.full(image.shape, 128, dtype=np.uint8)
    normalized = 0.5 + 0.42 * deviation / scale
    if display_noise_std > 0.0:
        rng = np.random.default_rng(stable_seed(label))
        normalized = normalized + rng.normal(0.0, display_noise_std, normalized.shape)
    return np.clip(255.0 * normalized, 0.0, 255.0).astype(np.uint8)


def particle(diameter_nm: float, z_nm: float, material: str = 'polystyrene') -> dict:
    material_properties = None
    if material == 'fluorescent_polystyrene':
        material_properties = {
            'fluorophore_density': 0.08,
            'excitation_peak_nm': 488.0,
            'emission_peak_nm': 520.0,
        }
    return {
        'name': f'{material}_{diameter_nm:g}nm',
        'motion': {'hydrodynamic_diameter_nm': float(diameter_nm), 'initial_position_nm': None},
        'signal_multiplier': 1.0,
        'components': [{
            'shape': 'sphere',
            'offset_nm': [0.0, 0.0, 0.0],
            'diameter_nm': float(diameter_nm),
            'material': material,
            'refractive_index': None,
            'signal_multiplier': 1.0,
            'material_properties': material_properties,
        }],
    }


def base_params(modality: str, diameter_nm: float, z_nm: float) -> dict:
    material = 'fluorescent_polystyrene' if 'fluorescence' in modality else 'polystyrene'
    params = deepcopy(PARAMS)
    params.update({
        'imaging_model': modality,
        'image_size_pixels': IMAGE_SIZE,
        'pixel_size_nm': PIXEL_SIZE_NM,
        'pupil_samples': PUPIL_SAMPLES,
        'psf_oversampling_factor': 2,
        'fps': 24.0,
        'duration_seconds': 1.0 / 24.0,
        'wavelength_nm': WAVELENGTH_NM,
        'numerical_aperture': 1.0,
        'refractive_index_medium': 1.33,
        'refractive_index_immersion': 1.518,
        'background_intensity': 1000.0,
        'read_noise_counts': 1.0,
        'camera_gain_e_per_count': 1.0,
        'shot_noise_enabled': False,
        'gaussian_noise_enabled': False,
        'fixed_pattern_gain_std': 0.0,
        'fixed_pattern_offset_counts': 0.0,
        'hot_pixel_fraction': 0.0,
        'scan_line_noise_counts': 0.0,
        'return_ideal_float_frames': True,
        'save_frame_sequence': False,
        'save_raw_frame_views': False,
        'mask_generation_enabled': False,
        'sample_environment_enabled': False,
        'sample_environment_pattern_enabled': False,
        'sample_environment_pattern': 'none',
        'background_subtraction_method': 'reference_frame',
        'z_stack_step_nm': 100.0,
        'initial_z_span_nm': max(4000.0, abs(float(z_nm)) * 2.0 + 1000.0),
        'particles': [particle(float(diameter_nm), float(z_nm), material=material)],
        'channels': None,
    })
    side_nm = IMAGE_SIZE * PIXEL_SIZE_NM
    params['particles'][0]['motion']['initial_position_nm'] = [0.5 * side_nm, 0.5 * side_nm, float(z_nm)]
    return params


def render(modality: str, diameter_nm: float = DIAMETER_NM, z_nm: float = Z_NM):
    params = base_params(modality, diameter_nm, z_nm)
    views = generate_single_frame_views(params)
    contrast = views.get('contrast_frame')
    signal_counts = views.get('ideal_signal_frame')
    if signal_counts is None:
        signal_counts = views.get('raw_signal_frame')
    detector_difference = views.get('detector_difference_frame')
    units = str(views.get('contrast_frame_units', 'detector_count_difference'))
    if contrast is None:
        raise RuntimeError(f'{modality} did not produce a contrast frame.')
    if signal_counts is None:
        raise RuntimeError(f'{modality} did not produce a detector-count frame.')
    contrast = np.asarray(contrast, dtype=float)
    if units == 'radians':
        budget_contrast = contrast
        measurement_model = 'phase'
    else:
        if detector_difference is None:
            budget_contrast = contrast
        else:
            budget_contrast = np.asarray(detector_difference, dtype=float)
        measurement_model = 'count'
    final = display_from_contrast(contrast, f'{modality}:{diameter_nm}:{z_nm}')
    return contrast, final, np.asarray(signal_counts, dtype=float), budget_contrast, measurement_model


def write_csv(path: Path, rows: list[dict]) -> None:
    if not rows:
        path.write_text('', encoding='utf-8')
        return
    with path.open('w', newline='', encoding='utf-8') as fh:
        writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


contrasts = {}
budget_contrasts = {}
count_images = {}
measurement_models = {}
for modality in HEADLINE_MODALITIES:
    print('render', modality)
    contrast, _, counts, budget_contrast, measurement_model = render(modality)
    contrasts[modality] = contrast
    budget_contrasts[modality] = budget_contrast
    count_images[modality] = counts
    measurement_models[modality] = measurement_model
noise = {m: NOISE_VARIANCE for m in contrasts}

# Particle-size sweep data.
diameters = np.geomspace(20.0, 200.0, 6)
size_rows = []
for modality in HEADLINE_MODALITIES:
    for diameter in diameters:
        contrast, _, _, _, _ = render(modality, diameter_nm=float(diameter))
        crlb = compute_localization_crlb(contrast, NOISE_VARIANCE, PIXEL_SIZE_NM)
        size_rows.append({'modality': modality, 'diameter_nm': float(diameter), 'sigma_xy_nm': float(crlb['sigma_xy_nm'])})
write_csv(OUTPUT_DIR / 'particle_size_sweep.csv', size_rows)

# Detected-quanta budget sweep data.
budgets = np.geomspace(1e3, 1e6, 6)
budget_rows = []
for modality in HEADLINE_MODALITIES:
    for budget in budgets:
        result = compare_modality_information_content_detected_quanta_normalized(
            {modality: budget_contrasts[modality]},
            pixel_size_nm=PIXEL_SIZE_NM,
            quanta_budget=float(budget),
            detected_count_image_by_modality={modality: count_images[modality]},
            measurement_model_by_modality={modality: measurement_models[modality]},
            readout_variance=BUDGET_READOUT_VARIANCE,
        )
        sigma = float(result['ranking_xy'][0][1])
        budget_rows.append({'modality': modality, 'budget': float(budget), 'sigma_xy_nm': sigma})
write_csv(OUTPUT_DIR / 'detected_quanta_budget_sweep.csv', budget_rows)

# Fusion curve data.
fusion_rows = []
for k in range(1, len(contrasts) + 1):
    result = compute_modality_fusion_crlb(contrasts, noise, PIXEL_SIZE_NM, subset_size=k)
    fusion_rows.append({
        'subset_size': k,
        'modalities_used': ';'.join(result['modalities_used']),
        'fusion_sigma_xy_nm': float(result['fusion_sigma_xy_nm']),
        'fusion_gain_xy': float(result['fusion_gain_xy']) if result.get('fusion_gain_xy') is not None else float('nan'),
    })
write_csv(OUTPUT_DIR / 'modality_fusion_crlb.csv', fusion_rows)

# Fisher-density arrays, one file per modality.
density_dir = OUTPUT_DIR / 'fisher_density_arrays'
density_dir.mkdir(parents=True, exist_ok=True)
density_rows = []
selected = [m for m in PANEL_MODALITIES if m in contrasts][:6]
for modality in selected:
    crlb = compute_localization_crlb(contrasts[modality], NOISE_VARIANCE, PIXEL_SIZE_NM, return_density=True)
    density = crlb['information_density_maps']['Ix_info_map'] + crlb['information_density_maps']['Iy_info_map']
    path = density_dir / f'{modality}_ixy_density.npy'
    np.save(path, density)
    density_rows.append({
        'modality': modality,
        'density_npy': str(path.relative_to(OUTPUT_DIR)),
        'density_min': float(np.nanmin(density)),
        'density_max': float(np.nanmax(density)),
        'density_sum': float(np.nansum(density)),
    })
write_csv(OUTPUT_DIR / 'fisher_density_manifest.csv', density_rows)


# Native-regime calibration checks. These are separate from the shared E03/E04
# matched-profile rankings: they test whether selected model paths behave
# sensibly in their own instrument regimes.
def render_with_params(params: dict):
    views = generate_single_frame_views(params)
    contrast = views.get('contrast_frame')
    signal_counts = views.get('ideal_signal_frame')
    reference_counts = views.get('ideal_reference_frame')
    if signal_counts is None:
        signal_counts = views.get('raw_signal_frame')
    if reference_counts is None:
        reference_counts = views.get('raw_reference_frame')
    if contrast is None or signal_counts is None:
        raise RuntimeError(f"{params.get('imaging_model')} did not produce contrast/count frames.")
    return views, np.asarray(contrast, dtype=float), np.asarray(signal_counts, dtype=float), None if reference_counts is None else np.asarray(reference_counts, dtype=float)


def native_params(modality: str, *, diameter_nm: float, z_nm: float = 0.0, material: str | None = None, pixel_size_nm: float = PIXEL_SIZE_NM, image_size_pixels: int = IMAGE_SIZE, overrides: dict | None = None) -> dict:
    params = base_params(modality, diameter_nm, z_nm)
    params['pixel_size_nm'] = float(pixel_size_nm)
    params['image_size_pixels'] = int(image_size_pixels)
    material_name = material or ('fluorescent_polystyrene' if 'fluorescence' in modality else 'polystyrene')
    params['particles'] = [particle(float(diameter_nm), float(z_nm), material=material_name)]
    side_nm = int(image_size_pixels) * float(pixel_size_nm)
    params['particles'][0]['motion']['initial_position_nm'] = [0.5 * side_nm, 0.5 * side_nm, float(z_nm)]
    params['shot_noise_enabled'] = True
    params['gaussian_noise_enabled'] = True
    params['read_noise_counts'] = 1.0
    params['camera_gain_e_per_count'] = 1.0
    if overrides:
        params.update(overrides)
    return params


def contrast_noise_for_views(signal_counts: np.ndarray, reference_counts: np.ndarray | None, params: dict) -> np.ndarray:
    return analysis_contrast_noise_variance(
        signal_counts,
        reference_counts,
        params,
        relative_reference=modality_uses_relative_reference_contrast(params.get('imaging_model', 'bright_field')),
    )


def summarize_native_case(case: dict) -> dict:
    params = native_params(
        case['modality'],
        diameter_nm=float(case['diameter_nm']),
        material=case.get('particle_material'),
        pixel_size_nm=float(case.get('pixel_size_nm', PIXEL_SIZE_NM)),
        image_size_pixels=int(case.get('image_size_pixels', IMAGE_SIZE)),
        overrides=case.get('overrides', {}),
    )
    views, contrast, signal_counts, reference_counts = render_with_params(params)
    noise_var = contrast_noise_for_views(signal_counts, reference_counts, params)
    crlb = compute_localization_crlb(contrast, noise_var, float(params['pixel_size_nm']))
    fisher = compute_fisher_information(contrast, noise_var, float(params['pixel_size_nm']))
    z_step_nm = float(case.get('z_step_nm', 0.0) or 0.0)
    sigma_z_nm = float('nan')
    axially_singular = True
    if z_step_nm > 0:
        minus_params = native_params(
            case['modality'],
            diameter_nm=float(case['diameter_nm']),
            z_nm=-z_step_nm,
            material=case.get('particle_material'),
            pixel_size_nm=float(case.get('pixel_size_nm', PIXEL_SIZE_NM)),
            image_size_pixels=int(case.get('image_size_pixels', IMAGE_SIZE)),
            overrides=case.get('overrides', {}),
        )
        plus_params = native_params(
            case['modality'],
            diameter_nm=float(case['diameter_nm']),
            z_nm=z_step_nm,
            material=case.get('particle_material'),
            pixel_size_nm=float(case.get('pixel_size_nm', PIXEL_SIZE_NM)),
            image_size_pixels=int(case.get('image_size_pixels', IMAGE_SIZE)),
            overrides=case.get('overrides', {}),
        )
        _, minus_contrast, _, _ = render_with_params(minus_params)
        _, plus_contrast, _, _ = render_with_params(plus_params)
        stack = np.stack([minus_contrast, contrast, plus_contrast], axis=0)
        crlb3 = compute_localization_crlb_3d(stack, noise_var, float(params['pixel_size_nm']), z_step_nm)
        sigma_z_nm = float(crlb3.get('sigma_z_nm', float('inf')))
        axially_singular = bool(crlb3.get('axially_singular', True))
    return {
        'case_id': case['case_id'],
        'modality': case['modality'],
        'display_name': modality_display_name(case['modality']),
        'regime_label': case['regime_label'],
        'measurement_model': case.get('measurement_model', 'count_reference_subtracted'),
        'particle_material': case.get('particle_material', ''),
        'diameter_nm': float(case['diameter_nm']),
        'pixel_size_nm': float(params['pixel_size_nm']),
        'image_size_pixels': int(params['image_size_pixels']),
        'z_nm': 0.0,
        'z_step_nm': z_step_nm,
        'noise_variance_source': 'camera_noise_propagated_to_contrast',
        'total_detected_quanta': float(np.nansum(signal_counts)),
        'mean_signal_counts': float(np.nanmean(signal_counts)),
        'mean_noise_variance': float(np.nanmean(noise_var)),
        'read_noise_counts': float(params.get('read_noise_counts', 0.0)),
        'camera_gain_e_per_count': float(params.get('camera_gain_e_per_count', 1.0)),
        'sigma_xy_nm': float(crlb['sigma_xy_nm']),
        'sigma_xy_pixels': float(crlb['sigma_xy_nm']) / float(params['pixel_size_nm']) if np.isfinite(crlb['sigma_xy_nm']) else float('inf'),
        'sigma_z_nm': sigma_z_nm,
        'axially_singular': bool(axially_singular),
        'fisher_trace': float(np.trace(fisher)),
        'fisher_det': float(np.linalg.det(fisher)),
        'finite_crlb': bool(np.isfinite(crlb['sigma_xy_nm'])),
        'param_overrides_json': json.dumps(case.get('overrides', {}), sort_keys=True, allow_nan=False),
        'notes': case.get('notes', ''),
    }


native_cases = [
    {
        'case_id': 'iscat_native_fresnel_dipole_gold40',
        'modality': 'interferometric',
        'regime_label': 'native iSCAT Fresnel reference plus high-NA dipole collection',
        'particle_material': 'gold',
        'diameter_nm': 40.0,
        'pixel_size_nm': 20.0,
        'image_size_pixels': 128,
        'z_step_nm': 100.0,
        'overrides': {
            'numerical_aperture': 1.3,
            'pupil_samples': 128,
            'refractive_index_medium': 1.33,
            'background_intensity': 7.5e3,
            'read_noise_counts': 0.5,
            'iscat_reference_model': 'fresnel',
            'iscat_reference_medium_material': 'water',
            'iscat_reference_substrate_material': 'glass',
            'iscat_collection_model': 'dipole_high_na',
        },
        'notes': 'Opt-in iSCAT reference-matched calibration path; Dong et al. normalize CRBs per collected scattered photon but do not report a detector background-count budget. Not part of shared ranking.',
    },
    {
        'case_id': 'tem_native_ctf_1nm_canvas',
        'modality': 'tem_phase_contrast',
        'regime_label': 'native TEM weak-phase CTF sampling',
        'particle_material': 'polystyrene',
        'diameter_nm': 100.0,
        'pixel_size_nm': 2.0,
        'image_size_pixels': 192,
        'z_step_nm': 10.0,
        'overrides': {
            'tem_dose_per_pixel': 1.0e4,
            'background_intensity': 1.0e4,
            'read_noise_counts': 1.0,
        },
        'notes': 'Native detector-pitch CTF check with oversampled model canvas; weak-phase proxy, not multislice TEM.',
    },
    {
        'case_id': 'sem_native_probe_5nm',
        'modality': 'sem_secondary_electron',
        'regime_label': 'native SEM scan/probe scale',
        'particle_material': 'gold',
        'diameter_nm': 100.0,
        'pixel_size_nm': 5.0,
        'image_size_pixels': 256,
        'z_step_nm': 20.0,
        'overrides': {
            'sem_probe_sigma_pixels': 1.0,
            'sem_electrons_per_pixel': 1.0e4,
            'background_intensity': 1.0e4,
            'read_noise_counts': 1.0,
        },
        'notes': 'Qualitative SEM proxy at native scan pitch; not Monte Carlo SEM.',
    },
    {
        'case_id': 'dpc_gain_noise_calibrated',
        'modality': 'differential_phase_contrast',
        'regime_label': 'DPC lower gain with propagated detector noise',
        'particle_material': 'polystyrene',
        'diameter_nm': 100.0,
        'pixel_size_nm': 65.0,
        'image_size_pixels': 192,
        'z_step_nm': 200.0,
        'overrides': {
            'dpc_phase_gradient_gain': 500.0,
            'background_intensity': 1000.0,
            'read_noise_counts': 2.0,
        },
        'notes': 'Sensitivity check for DPC gain/noise rather than sub-atomic unit-variance row.',
    },
    {
        'case_id': 'dark_field_native_gold_annular',
        'modality': 'dark_field',
        'regime_label': 'gold particle annular dark-field with count noise',
        'particle_material': 'gold',
        'diameter_nm': 100.0,
        'pixel_size_nm': 20.0,
        'image_size_pixels': 192,
        'z_step_nm': 100.0,
        'overrides': {
            'dark_field_illumination_count': 1.0e5,
            'dark_field_background_count': 10.0,
            'annular_dark_field_inner_sigma': 1.02,
            'annular_dark_field_outer_sigma': 1.45,
            'background_intensity': 1.0e5,
            'read_noise_counts': 1.0,
        },
        'notes': 'Dedicated dark-field-style check separate from clean polystyrene shared profile.',
    },
]

native_rows = []
for case in native_cases:
    print('native calibration', case['case_id'])
    native_rows.append(summarize_native_case(case))
write_csv(OUTPUT_DIR / 'native_regime_reference_checks.csv', native_rows)

sweep_rows = []
for budget in [3.0e3, 5.0e3, 7.5e3, 1.0e4, 2.5e4, 5.0e4, 1.0e5, 5.0e5]:
    case = deepcopy(native_cases[0])
    case['case_id'] = 'iscat_native_fresnel_dipole_gold40'
    case['overrides'] = dict(case['overrides'])
    case['overrides']['background_intensity'] = float(budget)
    row = summarize_native_case(case)
    sweep_rows.append({
        'case_id': case['case_id'],
        'modality': case['modality'],
        'sweep_parameter': 'background_intensity',
        'sweep_value': float(budget),
        'pixel_size_nm': row['pixel_size_nm'],
        'total_detected_quanta': row['total_detected_quanta'],
        'sigma_xy_nm': row['sigma_xy_nm'],
        'sigma_xy_pixels': row['sigma_xy_pixels'],
        'fisher_trace': row['fisher_trace'],
        'mean_noise_variance': row['mean_noise_variance'],
        'finite_crlb': row['finite_crlb'],
    })
write_csv(OUTPUT_DIR / 'native_regime_reference_sweeps.csv', sweep_rows)

manifest = {
    'schema_version': 'syniscopy-native-regime-reference-check-v1',
    'checks_csv': 'supplemental/outputs/E04/native_regime_reference_checks.csv',
    'sweeps_csv': 'supplemental/outputs/E04/native_regime_reference_sweeps.csv',
    'case_count': len(native_rows),
    'cases': native_cases,
}
(OUTPUT_DIR / 'native_regime_reference_check_manifest.json').write_text(json.dumps(manifest, indent=2, allow_nan=False), encoding='utf-8')
print('wrote native-regime reference checks:', OUTPUT_DIR / 'native_regime_reference_checks.csv')
calibration_rows = write_calibration_outputs(OUTPUT_DIR)
print('wrote registry native-regime reference-check table:', OUTPUT_DIR / 'calibration_reference_check_table.csv', 'rows:', len(calibration_rows))

print('wrote raw E04 sweep/fusion/density/native-calibration data:', OUTPUT_DIR)
